# 9 WorkFlow Analista Jr

### 9.1 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán
<br>El Analista Jr corre sus scripts en la virtual manchine **desktop-jr** que tiene estas características


*   Normal, paga tarifa completa, nunca es apagada por Google
*   reside en el datacenter de Sao Paulo, Brasil
*   64 GB de memoria RAM
*   8 vCPU



## 9.3  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Thu Aug 14 10:17:56 AM 2025"

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,657824,35.2,1443137,77.1,1443137,77.1
Vcells,1212826,9.3,8388608,64.0,1928907,14.8


In [3]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

Loading required package: data.table

Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: 'R.oo'


The following object is masked from 'package:R.methodsS3':

    throw


The following objects are masked from 'package:methods':

    getClasses, getMethods


The following objects are masked from 'package:base':

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: 'R.utils'


The following object is masked from 'package:utils':

    timestamp


The following objects are masked from 'package:base':

    cat, commandArgs, getOption, isOpen, nullfile, parse, use, warnings




#### Parametros

In [4]:
PARAM <- list()
PARAM$semilla_primigenia <- 100151

PARAM$experimento <- 9106
PARAM$dataset <- "analistajr_competencia_2025.csv.gz"

### 9.3.1   Preprocesamiento del dataset

#### 9.3.1.1  DT incorporar dataset

In [5]:
library(data.table)

dataset <- fread(
  "C:/Users/Luis/Documents/dm2025/Experimento colaborativo FEHistorico/datasets_analistajr_competencia_2025.csv.gz",
  encoding = "UTF-8"
)

print(dim(dataset))
print(head(dataset))

[1] 976374     55
   numero_de_cliente foto_mes internet cliente_edad cliente_antiguedad
               <int>    <int>    <int>        <int>              <int>
1:          29187730   201901        1           62                296
2:          29187961   201901        1           59                296
3:          29193101   201901        1           66                349
4:          29193281   201901        1           52                142
5:          29198891   201901        1           53                296
6:          29200770   201901        1           56                246
   mrentabilidad mrentabilidad_annual mcomisiones mactivos_margen
           <num>                <num>       <num>           <num>
1:      14301.42             43997.41     8970.82        -1262.22
2:       7542.75             34281.62     1960.95          -46.22
3:       5749.83             29663.24     4745.47        -3592.97
4:      12742.79             43614.97    12275.05         -718.19
5:       4237.90  

#### 9.3.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [6]:
if( !require("mice")) install.packages("mice", repos = "http://cran.us.r-project.org")
require("mice")

Loading required package: mice


Attaching package: 'mice'


The following object is masked from 'package:stats':

    filter


The following objects are masked from 'package:base':

    cbind, rbind




In [7]:
# Escrito por alumnos de  Universidad Austral  Rosario

Corregir_MICE <- function(pcampo, pmeses) {

  meth <- rep("", ncol(dataset))
  names(meth) <- colnames(dataset)
  meth[names(meth) == pcampo] <- "sample"

  # llamada a mice  !
  imputacion <- mice(dataset,
    method = meth,
    maxit = 5,
    m = 1,
    seed = 7)

  tbl <- mice::complete(dataset)

  dataset[, paste0(pcampo) := ifelse(foto_mes %in% pmeses, tbl[, get(pcampo)], get(pcampo))]

}

In [8]:
Corregir_interpolar <- function(pcampo, pmeses) {

  tbl <- dataset[, list(
    "v1" = shift(get(pcampo), 1, type = "lag"),
    "v2" = shift(get(pcampo), 1, type = "lead")
  ),
  by = eval(envg$PARAM$dataset_metadata$entity_id)
  ]

  tbl[, paste0(envg$PARAM$dataset_metadata$entity_id) := NULL]
  tbl[, promedio := rowMeans(tbl, na.rm = TRUE)]

  dataset[
    ,
    paste0(pcampo) := ifelse(!(foto_mes %in% pmeses),
      get(pcampo),
      tbl$promedio
    )
  ]
}

In [9]:
AsignarNA_campomeses <- function(pcampo, pmeses) {

  if( pcampo %in% colnames( dataset ) ) {

    dataset[ foto_mes %in% pmeses, paste0(pcampo) := NA ]
  }
}

In [10]:

Corregir_atributo <- function(pcampo, pmeses, pmetodo)
{
  # si el campo no existe en el dataset, Afuera !
  if( !(pcampo %in% colnames( dataset )) )
    return( 1 )

  # llamo a la funcion especializada que corresponde
  switch( pmetodo,
    "MachineLearning"     = AsignarNA_campomeses(pcampo, pmeses),
    "EstadisticaClasica"  = Corregir_interpolar(pcampo, pmeses),
    "MICE"                = Corregir_MICE(pcampo, pmeses),
  )

  return( 0 )
}

In [11]:

Corregir_Rotas <- function(dataset, pmetodo) {
  gc(verbose= FALSE)
  cat( "inicio Corregir_Rotas()\n")
  # acomodo los errores del dataset

  Corregir_atributo("active_quarter", c(202006), pmetodo) # 1
  Corregir_atributo("internet", c(202006), pmetodo) # 2

  Corregir_atributo("mrentabilidad", c(201905, 201910, 202006), pmetodo) # 3
  Corregir_atributo("mrentabilidad_annual", c(201905, 201910, 202006), pmetodo) # 4

  Corregir_atributo("mcomisiones", c(201905, 201910, 202006), pmetodo) # 5

  Corregir_atributo("mactivos_margen", c(201905, 201910, 202006), pmetodo) # 6
  Corregir_atributo("mpasivos_margen", c(201905, 201910, 202006), pmetodo) # 7

  Corregir_atributo("mcuentas_saldo", c(202006), pmetodo) # 8

  Corregir_atributo("ctarjeta_debito_transacciones", c(202006), pmetodo) # 9

  Corregir_atributo("mautoservicio", c(202006), pmetodo) # 10

  Corregir_atributo("ctarjeta_visa_transacciones", c(202006), pmetodo) # 11
  Corregir_atributo("mtarjeta_visa_consumo", c(202006), pmetodo) # 12

  Corregir_atributo("ctarjeta_master_transacciones", c(202006), pmetodo) # 13
  Corregir_atributo("mtarjeta_master_consumo", c(202006), pmetodo) # 14

  Corregir_atributo("ctarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 15
  Corregir_atributo("mttarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 16

  Corregir_atributo("ccajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 17

  Corregir_atributo("mcajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 18

  Corregir_atributo("ctarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 19

  Corregir_atributo("mtarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 20

  Corregir_atributo("ctarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 21

  Corregir_atributo("mtarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 22

  Corregir_atributo("ccomisiones_otras", c(201905, 201910, 202006), pmetodo) # 23
  Corregir_atributo("mcomisiones_otras", c(201905, 201910, 202006), pmetodo) # 24

  Corregir_atributo("cextraccion_autoservicio", c(202006), pmetodo) # 25
  Corregir_atributo("mextraccion_autoservicio", c(202006), pmetodo) # 26

  Corregir_atributo("ccheques_depositados", c(202006), pmetodo) # 27
  Corregir_atributo("mcheques_depositados", c(202006), pmetodo) # 28
  Corregir_atributo("ccheques_emitidos", c(202006), pmetodo) # 29
  Corregir_atributo("mcheques_emitidos", c(202006), pmetodo) # 30
  Corregir_atributo("ccheques_depositados_rechazados", c(202006), pmetodo) # 31
  Corregir_atributo("mcheques_depositados_rechazados", c(202006), pmetodo) # 32
  Corregir_atributo("ccheques_emitidos_rechazados", c(202006), pmetodo) # 33
  Corregir_atributo("mcheques_emitidos_rechazados", c(202006), pmetodo) # 34

  Corregir_atributo("tcallcenter", c(202006), pmetodo) # 35
  Corregir_atributo("ccallcenter_transacciones", c(202006), pmetodo) # 36

  Corregir_atributo("thomebanking", c(202006), pmetodo) # 37
  Corregir_atributo("chomebanking_transacciones", c(201910, 202006), pmetodo) # 38

  Corregir_atributo("ccajas_transacciones", c(202006), pmetodo) # 39
  Corregir_atributo("ccajas_consultas", c(202006), pmetodo) # 40

  Corregir_atributo("ccajas_depositos", c(202006, 202105), pmetodo) # 41

  Corregir_atributo("ccajas_extracciones", c(202006), pmetodo) # 41
  Corregir_atributo("ccajas_otras", c(202006), pmetodo) # 43

  Corregir_atributo("catm_trx", c(202006), pmetodo) # 44
  Corregir_atributo("matm", c(202006), pmetodo) # 45
  Corregir_atributo("catm_trx_other", c(202006), pmetodo) # 46
  Corregir_atributo("matm_other", c(202006), pmetodo) # 47

  cat( "fin Corregir_rotas()\n")
}


In [12]:
# resuelvo el Catastrophe Analysis

setorder( dataset, numero_de_cliente, foto_mes )

PARAM$CA$metodo= "MachineLearning"

if( PARAM$CA$metodo %in% c("MachineLearning", "EstadisticaClasica", "MICE") )
  Corregir_Rotas(dataset, PARAM$CA$metodo)

inicio Corregir_Rotas()
fin Corregir_rotas()


#### 9.3.1.3  DR  Data Drifting
Se intenta corregir el data drifting, ajustando por algunos indices financieros

In [13]:
# meses que me interesan para el ajuste de variables monetarias
vfoto_mes <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107, 202108, 202109
)


In [14]:
# los valores que siguen fueron calculados por alumnos

# momento 1.0  31-dic-2020 a las 23:59
vIPC <- c(
  1.9903030878, 1.9174403544, 1.8296186587,
  1.7728862972, 1.7212488323, 1.6776304408,
  1.6431248196, 1.5814483345, 1.4947526791,
  1.4484037589, 1.3913580777, 1.3404220402,
  1.3154288912, 1.2921698342, 1.2472681797,
  1.2300475145, 1.2118694724, 1.1881073259,
  1.1693969743, 1.1375456949, 1.1065619600,
  1.0681100000, 1.0370000000, 1.0000000000,
  0.9680542110, 0.9344152616, 0.8882274350,
  0.8532444140, 0.8251880213, 0.8003763543,
  0.7763107219, 0.7566381305, 0.7289384687
)

vdolar_blue <- c(
   39.045455,  38.402500,  41.639474,
   44.274737,  46.095455,  45.063333,
   43.983333,  54.842857,  61.059524,
   65.545455,  66.750000,  72.368421,
   77.477273,  78.191667,  82.434211,
  101.087500, 126.236842, 125.857143,
  130.782609, 133.400000, 137.954545,
  170.619048, 160.400000, 153.052632,
  157.900000, 149.380952, 143.615385,
  146.250000, 153.550000, 162.000000,
  178.478261, 180.878788, 184.357143
)

vdolar_oficial <- c(
   38.430000,  39.428000,  42.542105,
   44.354211,  46.088636,  44.955000,
   43.751429,  54.650476,  58.790000,
   61.403182,  63.012632,  63.011579,
   62.983636,  63.580556,  65.200000,
   67.872000,  70.047895,  72.520952,
   75.324286,  77.488500,  79.430909,
   83.134762,  85.484737,  88.181667,
   91.474000,  93.997778,  96.635909,
   98.526000,  99.613158, 100.619048,
  101.619048, 102.569048, 103.781818
)

vUVA <- c(
  2.001408838932958,  1.950325472789153,  1.89323032351521,
  1.8247220405493787, 1.746027787673673,  1.6871348409529485,
  1.6361678865622313, 1.5927529755859773, 1.5549162794128493,
  1.4949100586391746, 1.4197729500774545, 1.3678188186372326,
  1.3136508617223726, 1.2690535173062818, 1.2381595983200178,
  1.211656735577568,  1.1770808941405335, 1.1570338657445522,
  1.1388769475653255, 1.1156993751209352, 1.093638313080772,
  1.0657171590878205, 1.0362173587708712, 1.0,
  0.9669867858358365, 0.9323750098728378, 0.8958202912590305,
  0.8631993702994263, 0.8253893405524657, 0.7928918905364516,
  0.7666323845128089, 0.7428976357662823, 0.721615762047849
)


In [15]:
tb_indices <- as.data.table( list(
  "IPC" = vIPC,
  "dolar_blue" = vdolar_blue,
  "dolar_oficial" = vdolar_oficial,
  "UVA" = vUVA
  )
)

tb_indices[[ 'foto_mes' ]] <- vfoto_mes

tb_indices

IPC,dolar_blue,dolar_oficial,UVA,foto_mes
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1.9903031,39.04545,38.43000,2.0014088,201901
1.9174404,38.40250,39.42800,1.9503255,201902
1.8296187,41.63947,42.54210,1.8932303,201903
1.7728863,44.27474,44.35421,1.8247220,201904
1.7212488,46.09546,46.08864,1.7460278,201905
1.6776304,45.06333,44.95500,1.6871348,201906
1.6431248,43.98333,43.75143,1.6361679,201907
1.5814483,54.84286,54.65048,1.5927530,201908
1.4947527,61.05952,58.79000,1.5549163,201909


In [16]:
drift_UVA <- function(campos_monetarios) {
  cat( "inicio drift_UVA()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.UVA,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_UVA()\n")
}


In [17]:
drift_dolar_oficial <- function(campos_monetarios) {
  cat( "inicio drift_dolar_oficial()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_oficial,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_oficial()\n")
}


In [18]:
drift_dolar_blue <- function(campos_monetarios) {
  cat( "inicio drift_dolar_blue()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_blue,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_blue()\n")
}


In [19]:
drift_deflacion <- function(campos_monetarios) {
  cat( "inicio drift_deflacion()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.IPC,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_deflacion()\n")
}


In [20]:
drift_rank_simple <- function(campos_drift) {

  cat( "inicio drift_rank_simple()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_rank") :=
      (frank(get(campo), ties.method = "random") - 1) / (.N - 1), by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat( "fin drift_rank_simple()\n")
}


In [21]:
# El cero se transforma en cero
# los positivos se rankean por su lado
# los negativos se rankean por su lado

drift_rank_cero_fijo <- function(campos_drift) {

  cat( "inicio drift_rank_cero_fijo()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[get(campo) == 0, paste0(campo, "_rank") := 0]
    dataset[get(campo) > 0, paste0(campo, "_rank") :=
      frank(get(campo), ties.method = "random") / .N, by = list(foto_mes)]

    dataset[get(campo) < 0, paste0(campo, "_rank") :=
      -frank(-get(campo), ties.method = "random") / .N, by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat("\n")
  cat( "fin drift_rank_cero_fijo()\n")
}


In [22]:
drift_estandarizar <- function(campos_drift) {

  cat( "inicio drift_estandarizar()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_normal") :=
      (get(campo) -mean(campo, na.rm=TRUE)) / sd(get(campo), na.rm=TRUE),
      by = list(foto_mes)]

    dataset[, (campo) := NULL]
  }
  cat( "fin drift_estandarizar()\n")
}


In [23]:
# por como armé los nombres de campos,
#  estos son los campos que expresan variables monetarias
campos_monetarios <- colnames(dataset)
campos_monetarios <- campos_monetarios[campos_monetarios %like%
  "^(m|Visa_m|Master_m|vm_m)"]

campos_monetarios

[1] "mrentabilidad"                      "mrentabilidad_annual"              
 [3] "mcomisiones"                        "mactivos_margen"                   
 [5] "mpasivos_margen"                    "mcuenta_corriente"                 
 [7] "mcaja_ahorro"                       "mcuentas_saldo"                    
 [9] "mtarjeta_visa_consumo"              "mtarjeta_master_consumo"           
[11] "mprestamos_personales"              "mpayroll"                          
[13] "mttarjeta_visa_debitos_automaticos" "mcomisiones_mantenimiento"         
[15] "mtransferencias_recibidas"          "Master_mfinanciacion_limite"       
[17] "Master_msaldototal"                 "Master_mlimitecompra"              
[19] "Master_mconsumototal"               "Master_mpagominimo"                
[21] "Visa_mfinanciacion_limite"          "Visa_msaldototal"                  
[23] "Visa_mlimitecompra"                 "Visa_mconsumototal"                
[25] "Visa_mpagominimo"

In [24]:
# ejecuto el Data Drifting
setorder( dataset, numero_de_cliente, foto_mes )


PARAM$DR$metodo <- "deflacion"

switch(PARAM$DR$metodo,
  "ninguno"        = cat("No hay correccion del data drifting"),
  "rank_simple"    = drift_rank_simple(campos_monetarios),
  "rank_cero_fijo" = drift_rank_cero_fijo(campos_monetarios),
  "deflacion"      = drift_deflacion(campos_monetarios),
  "dolar_blue"     = drift_dolarblue(campos_monetarios),
  "dolar_oficial"  = drift_dolaroficial(campos_monetarios),
  "UVA"            = drift_UVA(campos_monetarios),
  "estandarizar"   = drift_estandarizar(campos_monetarios)
)


inicio drift_deflacion()
fin drift_deflacion()


In [25]:
colnames(dataset)

[1] "numero_de_cliente"                  "foto_mes"                          
 [3] "internet"                           "cliente_edad"                      
 [5] "cliente_antiguedad"                 "mrentabilidad"                     
 [7] "mrentabilidad_annual"               "mcomisiones"                       
 [9] "mactivos_margen"                    "mpasivos_margen"                   
[11] "cproductos"                         "mcuenta_corriente"                 
[13] "mcaja_ahorro"                       "cdescubierto_preacordado"          
[15] "mcuentas_saldo"                     "ctarjeta_visa"                     
[17] "ctarjeta_visa_transacciones"        "mtarjeta_visa_consumo"             
[19] "ctarjeta_master"                    "ctarjeta_master_transacciones"     
[21] "mtarjeta_master_consumo"            "cprestamos_personales"             
[23] "mprestamos_personales"              "cpayroll_trx"                      
[25] "mpayroll"                           "mttarjeta_visa_debitos_automaticos"
[27] "ccomisiones_mantenimiento"          "mcomisiones_mantenimiento"         
[29] "ccomisiones_otras"                  "mtransferencias_recibidas"         
[31] "ccallcenter_transacciones"          "thomebanking"                      
[33] "chomebanking_transacciones"         "ctrx_quarter"                      
[35] "Master_status"                      "Master_mfinanciacion_limite"       
[37] "Master_Fvencimiento"                "Master_msaldototal"                
[39] "Master_mlimitecompra"               "Master_fultimo_cierre"             
[41] "Master_fechaalta"                   "Master_mconsumototal"              
[43] "Master_cconsumos"                   "Master_mpagominimo"                
[45] "Visa_status"                        "Visa_mfinanciacion_limite"         
[47] "Visa_Fvencimiento"                  "Visa_msaldototal"                  
[49] "Visa_mlimitecompra"                 "Visa_fultimo_cierre"               
[51] "Visa_fechaalta"                     "Visa_mconsumototal"                
[53] "Visa_cconsumos"                     "Visa_mpagominimo"                  
[55] "clase_ternaria"

In [26]:
# se intenta corregir el data drifting utilizando algunos indices financieros

#### 9.3.1.3  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

In [27]:
# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

# el mes 1,2, ..12
if( atributos_presentes( c("foto_mes") ))
  dataset[, kmes := foto_mes %% 100]

# variable extraida de una tesis de maestria de Irlanda
if( atributos_presentes( c("mpayroll", "cliente_edad") ))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]


In [28]:
# visualizo las columas del dataset a esta etapa
colnames(dataset)

[1] "numero_de_cliente"                  "foto_mes"                          
 [3] "internet"                           "cliente_edad"                      
 [5] "cliente_antiguedad"                 "mrentabilidad"                     
 [7] "mrentabilidad_annual"               "mcomisiones"                       
 [9] "mactivos_margen"                    "mpasivos_margen"                   
[11] "cproductos"                         "mcuenta_corriente"                 
[13] "mcaja_ahorro"                       "cdescubierto_preacordado"          
[15] "mcuentas_saldo"                     "ctarjeta_visa"                     
[17] "ctarjeta_visa_transacciones"        "mtarjeta_visa_consumo"             
[19] "ctarjeta_master"                    "ctarjeta_master_transacciones"     
[21] "mtarjeta_master_consumo"            "cprestamos_personales"             
[23] "mprestamos_personales"              "cpayroll_trx"                      
[25] "mpayroll"                           "mttarjeta_visa_debitos_automaticos"
[27] "ccomisiones_mantenimiento"          "mcomisiones_mantenimiento"         
[29] "ccomisiones_otras"                  "mtransferencias_recibidas"         
[31] "ccallcenter_transacciones"          "thomebanking"                      
[33] "chomebanking_transacciones"         "ctrx_quarter"                      
[35] "Master_status"                      "Master_mfinanciacion_limite"       
[37] "Master_Fvencimiento"                "Master_msaldototal"                
[39] "Master_mlimitecompra"               "Master_fultimo_cierre"             
[41] "Master_fechaalta"                   "Master_mconsumototal"              
[43] "Master_cconsumos"                   "Master_mpagominimo"                
[45] "Visa_status"                        "Visa_mfinanciacion_limite"         
[47] "Visa_Fvencimiento"                  "Visa_msaldototal"                  
[49] "Visa_mlimitecompra"                 "Visa_fultimo_cierre"               
[51] "Visa_fechaalta"                     "Visa_mconsumototal"                
[53] "Visa_cconsumos"                     "Visa_mpagominimo"                  
[55] "clase_ternaria"                     "kmes"                              
[57] "mpayroll_sobre_edad"

#### 9.3.1.4  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest

In [29]:
if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

Loading required package: lightgbm



In [30]:
AgregaVarRandomForest <- function() {

  cat( "inicio AgregaVarRandomForest()\n")
  gc(verbose= FALSE)
  dataset[, clase01 := 0L ]
  dataset[ clase_ternaria %in% PARAM$FE_rf$train$clase01_valor1,
      clase01 := 1L ]

  campos_buenos <- setdiff(
    colnames(dataset),
    c( "clase_ternaria", "clase01")
  )

  dataset[, entrenamiento :=
    as.integer( foto_mes %in% PARAM$FE_rf$train$training )]

  dtrain <- lgb.Dataset(
    data = data.matrix(dataset[entrenamiento == TRUE, campos_buenos, with = FALSE]),
    label = dataset[entrenamiento == TRUE, clase01],
    free_raw_data = FALSE
  )

  modelo <- lgb.train(
     data = dtrain,
     param = PARAM$FE_rf$lgb_param,
     verbose = -100
  )

  cat( "Fin construccion RandomForest\n" )
  # grabo el modelo, achivo .model
  lgb.save(modelo, file="modelo.model" )

  qarbolitos <- copy(PARAM$FE_rf$lgb_param$num_iterations)

  periodos <- dataset[ , unique( foto_mes ) ]

  for( periodo in  periodos )
  {
    cat( "periodo = ", periodo, "\n" )
    datamatrix <- data.matrix(dataset[ foto_mes== periodo, campos_buenos, with = FALSE])

    cat( "Inicio prediccion\n" )
    prediccion <- predict(
        modelo,
        datamatrix,
        type = "leaf"
    )
    cat( "Fin prediccion\n" )

    for( arbolito in 1:qarbolitos )
    {
       cat( arbolito, " " )
       hojas_arbol <- unique(prediccion[ , arbolito])

       for (pos in 1:length(hojas_arbol)) {
         # el numero de nodo de la hoja, estan salteados
         nodo_id <- hojas_arbol[pos]
         dataset[ foto_mes== periodo, paste0(
            "rf_", sprintf("%03d", arbolito),
             "_", sprintf("%03d", nodo_id)
          ) :=  as.integer( nodo_id == prediccion[ , arbolito]) ]

       }

       rm( hojas_arbol )
    }
    cat( "\n" )

    rm( prediccion )
    rm( datamatrix )
    gc(verbose= FALSE)
  }

  gc(verbose= FALSE)

  # borro clase01 , no debe ensuciar el dataset
  dataset[ , clase01 := NULL ]

}


In [31]:
# Parametros de Feature Engineering  a partir de hojas de Random Forest

# Estos CUATRO parametros son los que se deben modificar
PARAM$FE_rf$arbolitos= 20
PARAM$FE_rf$hojas_por_arbol= 16
PARAM$FE_rf$datos_por_hoja= 100
PARAM$FE_rf$mtry_ratio= 0.2

# Estos son quasi fijos
PARAM$FE_rf$train$clase01_valor1 <- c( "BAJA+2", "BAJA+1")
PARAM$FE_rf$train$training <- c( 202101, 202102, 202103)

# Estos TAMBIEN son quasi fijos
PARAM$FE_rf$lgb_param <-list(
    # parametros que se pueden cambiar
    num_iterations = PARAM$FE_rf$arbolitos,
    num_leaves  = PARAM$FE_rf$hojas_por_arbol,
    min_data_in_leaf = PARAM$FE_rf$datos_por_hoja,
    feature_fraction_bynode  = PARAM$FE_rf$mtry_ratio,

    # para que LightGBM emule Random Forest
    boosting = "rf",
    bagging_fraction = ( 1.0 - 1.0/exp(1.0) ),
    bagging_freq = 1.0,
    feature_fraction = 1.0,

    # genericos de LightGBM
    max_bin = 31L,
    objective = "binary",
    first_metric_only = TRUE,
    boost_from_average = TRUE,
    feature_pre_filter = FALSE,
    force_row_wise = TRUE,
    verbosity = -100,
    max_depth = -1L,
    min_gain_to_split = 0.0,
    min_sum_hessian_in_leaf = 0.001,
    lambda_l1 = 0.0,
    lambda_l2 = 0.0,

    pos_bagging_fraction = 1.0,
    neg_bagging_fraction = 1.0,
    is_unbalance = FALSE,
    scale_pos_weight = 1.0,

    drop_rate = 0.1,
    max_drop = 50,
    skip_drop = 0.5,

    extra_trees = FALSE
  )

In [32]:
# Feature Engineering agregando variables de Random Forest
AgregaVarRandomForest()

inicio AgregaVarRandomForest()
Fin construccion RandomForest
periodo =  202003 
Inicio prediccion
Fin prediccion
1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  16  17  18  19  20  
periodo =  202004 
Inicio prediccion
Fin prediccion
1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  16  17  18  19  20  
periodo =  202005 
Inicio prediccion
Fin prediccion
1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  16  17  18  19  20  
periodo =  202006 
Inicio prediccion
Fin prediccion
1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  16  17  18  19  20  
periodo =  202007 
Inicio prediccion
Fin prediccion
1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  16  17  18  19  20  
periodo =  202008 
Inicio prediccion
Fin prediccion
1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  16  17  18  19  20  
periodo =  202009 
Inicio prediccion
Fin prediccion
1  2  3  4  5  6  7  8  9  10  11  12  13  14  15  16  17  18  19  20  
periodo =  202010 
Inicio prediccion
Fin prediccion
1  2  3  4  5  6  7

In [33]:
ncol(dataset)
colnames(dataset)

[1] 378

[1] "numero_de_cliente"                  "foto_mes"                          
  [3] "internet"                           "cliente_edad"                      
  [5] "cliente_antiguedad"                 "mrentabilidad"                     
  [7] "mrentabilidad_annual"               "mcomisiones"                       
  [9] "mactivos_margen"                    "mpasivos_margen"                   
 [11] "cproductos"                         "mcuenta_corriente"                 
 [13] "mcaja_ahorro"                       "cdescubierto_preacordado"          
 [15] "mcuentas_saldo"                     "ctarjeta_visa"                     
 [17] "ctarjeta_visa_transacciones"        "mtarjeta_visa_consumo"             
 [19] "ctarjeta_master"                    "ctarjeta_master_transacciones"     
 [21] "mtarjeta_master_consumo"            "cprestamos_personales"             
 [23] "mprestamos_personales"              "cpayroll_trx"                      
 [25] "mpayroll"                           "mttarjeta_visa_debitos_automaticos"
 [27] "ccomisiones_mantenimiento"          "mcomisiones_mantenimiento"         
 [29] "ccomisiones_otras"                  "mtransferencias_recibidas"         
 [31] "ccallcenter_transacciones"          "thomebanking"                      
 [33] "chomebanking_transacciones"         "ctrx_quarter"                      
 [35] "Master_status"                      "Master_mfinanciacion_limite"       
 [37] "Master_Fvencimiento"                "Master_msaldototal"                
 [39] "Master_mlimitecompra"               "Master_fultimo_cierre"             
 [41] "Master_fechaalta"                   "Master_mconsumototal"              
 [43] "Master_cconsumos"                   "Master_mpagominimo"                
 [45] "Visa_status"                        "Visa_mfinanciacion_limite"         
 [47] "Visa_Fvencimiento"                  "Visa_msaldototal"                  
 [49] "Visa_mlimitecompra"                 "Visa_fultimo_cierre"               
 [51] "Visa_fechaalta"                     "Visa_mconsumototal"                
 [53] "Visa_cconsumos"                     "Visa_mpagominimo"                  
 [55] "clase_ternaria"                     "kmes"                              
 [57] "mpayroll_sobre_edad"                "entrenamiento"                     
 [59] "rf_001_006"                         "rf_001_008"                        
 [61] "rf_001_007"                         "rf_001_015"                        
 [63] "rf_001_001"                         "rf_001_012"                        
 [65] "rf_001_000"                         "rf_001_010"                        
 [67] "rf_001_011"                         "rf_001_003"                        
 [69] "rf_001_009"                         "rf_001_014"                        
 [71] "rf_001_005"                         "rf_001_002"                        
 [73] "rf_001_013"                         "rf_001_004"                        
 [75] "rf_002_009"                         "rf_002_014"                        
 [77] "rf_002_002"                         "rf_002_010"                        
 [79] "rf_002_006"                         "rf_002_003"                        
 [81] "rf_002_012"                         "rf_002_011"                        
 [83] "rf_002_008"                         "rf_002_013"                        
 [85] "rf_002_005"                         "rf_002_001"                        
 [87] "rf_002_015"                         "rf_002_004"                        
 [89] "rf_002_000"                         "rf_002_007"                        
 [91] "rf_003_005"                         "rf_003_007"                        
 [93] "rf_003_011"                         "rf_003_014"                        
 [95] "rf_003_006"                         "rf_003_001"                        
 [97] "rf_003_013"                         "rf_003_010"                        
 [99] "rf_003_012"                         "rf_003_004"                        
[1

In [34]:
# No se implementa Feature Engineering a partir de Random Forest

#### 9.3.1.5  FEhist Feature Engineering historico

El Fature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

In [35]:
if( !require("Rcpp")) install.packages("Rcpp", repos = "http://cran.us.r-project.org")
require("Rcpp")

Loading required package: Rcpp



In [36]:
# se calculan para los 6 meses previos el minimo, maximo y
#  tendencia calculada con cuadrados minimos
# la formula de calculo de la tendencia puede verse en
#  https://stats.libretexts.org/Bookshelves/Introductory_Statistics/Book%3A_Introductory_Statistics_(Shafer_and_Zhang)/10%3A_Correlation_and_Regression/10.04%3A_The_Least_Squares_Regression_Line
# para la maxíma velocidad esta funcion esta escrita en lenguaje C,
# y no en la porqueria de R o Python

cppFunction("NumericVector fhistC(NumericVector pcolumna, IntegerVector pdesde )
{
  /* Aqui se cargan los valores para la regresion */
  double  x[100] ;
  double  y[100] ;

  int n = pcolumna.size();
  NumericVector out( 5*n );

  for(int i = 0; i < n; i++)
  {
    //lag
    if( pdesde[i]-1 < i )  out[ i + 4*n ]  =  pcolumna[i-1] ;
    else                   out[ i + 4*n ]  =  NA_REAL ;


    int  libre    = 0 ;
    int  xvalor   = 1 ;

    for( int j= pdesde[i]-1;  j<=i; j++ )
    {
       double a = pcolumna[j] ;

       if( !R_IsNA( a ) )
       {
          y[ libre ]= a ;
          x[ libre ]= xvalor ;
          libre++ ;
       }

       xvalor++ ;
    }

    /* Si hay al menos dos valores */
    if( libre > 1 )
    {
      double  xsum  = x[0] ;
      double  ysum  = y[0] ;
      double  xysum = xsum * ysum ;
      double  xxsum = xsum * xsum ;
      double  vmin  = y[0] ;
      double  vmax  = y[0] ;

      for( int h=1; h<libre; h++)
      {
        xsum  += x[h] ;
        ysum  += y[h] ;
        xysum += x[h]*y[h] ;
        xxsum += x[h]*x[h] ;

        if( y[h] < vmin )  vmin = y[h] ;
        if( y[h] > vmax )  vmax = y[h] ;
      }

      out[ i ]  =  (libre*xysum - xsum*ysum)/(libre*xxsum -xsum*xsum) ;
      out[ i + n ]    =  vmin ;
      out[ i + 2*n ]  =  vmax ;
      out[ i + 3*n ]  =  ysum / libre ;
    }
    else
    {
      out[ i       ]  =  NA_REAL ;
      out[ i + n   ]  =  NA_REAL ;
      out[ i + 2*n ]  =  NA_REAL ;
      out[ i + 3*n ]  =  NA_REAL ;
    }
  }

  return  out;
}")


In [37]:
# calcula la tendencia de las variables cols de los ultimos 6 meses
# la tendencia es la pendiente de la recta que ajusta por cuadrados minimos
# La funcionalidad de ratioavg es autoria de  Daiana Sparta,  UAustral  2021

TendenciaYmuchomas <- function(
    dataset, cols, ventana = 6, tendencia = TRUE,
    minimo = TRUE, maximo = TRUE, promedio = TRUE,
    ratioavg = FALSE, ratiomax = FALSE) {
  gc(verbose= FALSE)
  # Esta es la cantidad de meses que utilizo para la historia
  ventana_regresion <- ventana

  last <- nrow(dataset)

  # creo el vector_desde que indica cada ventana
  # de esta forma se acelera el procesamiento ya que lo hago una sola vez
  vector_ids <- dataset[ , numero_de_cliente ]

  vector_desde <- seq(
    -ventana_regresion + 2,
    nrow(dataset) - ventana_regresion + 1
  )

  vector_desde[1:ventana_regresion] <- 1

  for (i in 2:last) {
    if (vector_ids[i - 1] != vector_ids[i]) {
      vector_desde[i] <- i
    }
  }
  for (i in 2:last) {
    if (vector_desde[i] < vector_desde[i - 1]) {
      vector_desde[i] <- vector_desde[i - 1]
    }
  }

  for (campo in cols) {
    nueva_col <- fhistC(dataset[, get(campo)], vector_desde)

    if (tendencia) {
      dataset[, paste0(campo, "_tend", ventana) :=
        nueva_col[(0 * last + 1):(1 * last)]]
    }

    if (minimo) {
      dataset[, paste0(campo, "_min", ventana) :=
        nueva_col[(1 * last + 1):(2 * last)]]
    }

    if (maximo) {
      dataset[, paste0(campo, "_max", ventana) :=
        nueva_col[(2 * last + 1):(3 * last)]]
    }

    if (promedio) {
      dataset[, paste0(campo, "_avg", ventana) :=
        nueva_col[(3 * last + 1):(4 * last)]]
    }

    if (ratioavg) {
      dataset[, paste0(campo, "_ratioavg", ventana) :=
        get(campo) / nueva_col[(3 * last + 1):(4 * last)]]
    }

    if (ratiomax) {
      dataset[, paste0(campo, "_ratiomax", ventana) :=
        get(campo) / nueva_col[(2 * last + 1):(3 * last)]]
    }
  }
}

In [38]:
# parametros de Feature Engineering Historico de lags
PARAM$FEhist$usar_lags <- FALSE # TRUE: activa uso de lags; FALSE caso contrario.
PARAM$FEhist$lag_config <- list(
    lag1_activo = FALSE, # TRUE: activa lag1; FALSE caso contrario.
    lag2_activo = FALSE, # TRUE: activa lag2; FALSE caso contrario.
    lag3_activo = FALSE  # TRUE: activa lag3; FALSE caso contrario.
)

In [39]:
# parametros de Feature Engineering Historico de Tendencias
PARAM$FE_hist$Tendencias$run <- TRUE # esto activa las tendencias (usar_tendencias)
PARAM$FE_hist$Tendencias$ventana <- 6 # cantidad de meses: 6 o 12
PARAM$FE_hist$Tendencias$tendencia <- TRUE # ventana de tendencia segun la cantidad de meses
PARAM$FE_hist$Tendencias$minimo <- TRUE # ventana de minimo segun la cantidad de meses
PARAM$FE_hist$Tendencias$maximo <- TRUE # ventana de maximo segun la cantidad de meses
PARAM$FE_hist$Tendencias$promedio <- FALSE # ventana de promedio segun la cantidad de meses
PARAM$FE_hist$Tendencias$ratioavg <- FALSE # ventana del rationavg segun la cantidad de meses
PARAM$FE_hist$Tendencias$ratiomax <- FALSE # ventanta del ratiomax segun la cantidad de meses

In [40]:
# Feature Engineering Historico

setorder(dataset, numero_de_cliente, foto_mes)

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
) )

if (isTRUE(PARAM$FEhist$usar_lags)) {
    
    # Lag 1
    if (isTRUE(PARAM$FEhist$lag_config$lag1_activo)) {
        dataset[,
            paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
            by = numero_de_cliente,
            .SDcols = cols_lagueables
        ]
        for (vcol in cols_lagueables) {
            dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
        }
    }
    
    # Lag 2
    if (isTRUE(PARAM$FEhist$lag_config$lag2_activo)) {
        dataset[,
            paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
            by = numero_de_cliente,
            .SDcols = cols_lagueables
        ]
        for (vcol in cols_lagueables) {
            dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
        }
    }
    
    # Lag 3
    if (isTRUE(PARAM$FEhist$lag_config$lag3_activo)) {
        dataset[,
            paste0(cols_lagueables, "_lag3") := shift(.SD, 3, NA, "lag"),
            by = numero_de_cliente,
            .SDcols = cols_lagueables
        ]
        for (vcol in cols_lagueables) {
            dataset[, paste0(vcol, "_delta3") := get(vcol) - get(paste0(vcol, "_lag3"))]
        }
    }
}

In [41]:
# Identificar columnas base ANTES de cualquier procesamiento
cols_base <- copy(setdiff(colnames(dataset), c("numero_de_cliente", "foto_mes", "clase_ternaria")))
columnas_control <- c("numero_de_cliente", "foto_mes", "clase_ternaria")
columnas_originales <- c(columnas_control, cols_base)

cat("=== CONFIGURACIÓN INICIAL ===\n")
cat("Total columnas originales:", length(columnas_originales), "\n")
cat("Columnas base para procesar:", length(cols_base), "\n")

# Crear dos copias independientes del dataset original
cat("\n=== CREANDO COPIAS INDEPENDIENTES ===\n")
dataset_copia_6 <- copy(dataset[, ..columnas_originales])
dataset_copia_12 <- copy(dataset[, ..columnas_originales])

cat("Copia para ventana 6 - Columnas:", ncol(dataset_copia_6), "\n")
cat("Copia para ventana 12 - Columnas:", ncol(dataset_copia_12), "\n")

# Procesar ventana 6 en su copia independiente
if (PARAM$FE_hist$Tendencias$run) {
  cat("\n=== PROCESANDO VENTANA 6 ===\n")
  setorder(dataset_copia_6, numero_de_cliente, foto_mes)
  
  TendenciaYmuchomas(
    dataset_copia_6,
    cols = cols_base,
    ventana = PARAM$FE_hist$Tendencias$ventana,
    tendencia = PARAM$FE_hist$Tendencias$tendencia,
    minimo = PARAM$FE_hist$Tendencias$minimo,
    maximo = PARAM$FE_hist$Tendencias$maximo,
    promedio = PARAM$FE_hist$Tendencias$promedio,
    ratioavg = PARAM$FE_hist$Tendencias$ratioavg,
    ratiomax = PARAM$FE_hist$Tendencias$ratiomax
  )
  
  cat("Ventana 6 completada - Columnas:", ncol(dataset_copia_6), "\n")
}

# Procesar ventana 12 en su copia independiente
if (PARAM$FE_hist$Tendencias$run) {
  cat("\n=== PROCESANDO VENTANA 12 ===\n")
  setorder(dataset_copia_12, numero_de_cliente, foto_mes)

  PARAM$FE_hist$Tendencias$ventana <- 12 # ACÁ INCORPORO LAS DE 12 meses
  TendenciaYmuchomas(
    dataset_copia_12,
    cols = cols_base,  # Exactamente las mismas columnas base
    ventana = PARAM$FE_hist$Tendencias$ventana,
    tendencia = PARAM$FE_hist$Tendencias$tendencia,
    minimo = PARAM$FE_hist$Tendencias$minimo,
    maximo = PARAM$FE_hist$Tendencias$maximo,
    promedio = PARAM$FE_hist$Tendencias$promedio,
    ratioavg = PARAM$FE_hist$Tendencias$ratioavg,
    ratiomax = PARAM$FE_hist$Tendencias$ratiomax
  )
  
  cat("Ventana 12 completada - Columnas:", ncol(dataset_copia_12), "\n")
}

# Combinar resultados en el dataset original
cat("\n=== COMBINANDO RESULTADOS ===\n")

# Extraer solo las columnas nuevas de cada copia
cols_nuevas_6 <- setdiff(colnames(dataset_copia_6), columnas_originales)
cols_nuevas_12 <- setdiff(colnames(dataset_copia_12), columnas_originales)

cat("Columnas nuevas de ventana 6:", length(cols_nuevas_6), "\n")
cat("Columnas nuevas de ventana 12:", length(cols_nuevas_12), "\n")

# Verificar que no hay conflictos de nombres (no debería haberlos)
conflictos <- intersect(cols_nuevas_6, cols_nuevas_12)
if(length(conflictos) > 0) {
  cat("WARNING: Conflictos de nombres:", length(conflictos), "\n")
  cat("Conflictos:", paste(head(conflictos, 5), collapse = ", "), "\n")
}

# Transferir columnas de ventana 6 al dataset original
if(length(cols_nuevas_6) > 0) {
  for(col in cols_nuevas_6) {
    dataset[, (col) := dataset_copia_6[[col]]]
  }
  cat("Columnas de ventana 6 transferidas al dataset original\n")
}

# Transferir columnas de ventana 12 al dataset original  
if(length(cols_nuevas_12) > 0) {
  for(col in cols_nuevas_12) {
    dataset[, (col) := dataset_copia_12[[col]]]
  }
  cat("Columnas de ventana 12 transferidas al dataset original\n")
}

# Limpiar memoria
rm(dataset_copia_6, dataset_copia_12)
gc()

=== CONFIGURACIÓN INICIAL ===
Total columnas originales: 378 
Columnas base para procesar: 375 

=== CREANDO COPIAS INDEPENDIENTES ===
Copia para ventana 6 - Columnas: 378 
Copia para ventana 12 - Columnas: 378 

=== PROCESANDO VENTANA 6 ===
Ventana 6 completada - Columnas: 1402 

=== PROCESANDO VENTANA 12 ===
Ventana 12 completada - Columnas: 1402 

=== COMBINANDO RESULTADOS ===
Columnas nuevas de ventana 6: 1024 
Columnas nuevas de ventana 12: 1024 


ERROR: Error: cannot allocate vector of size 7.4 Mb


Verificacion de los campos recien creados

In [44]:
ncol(dataset)
colnames(dataset)

[1] 2426

[1] "numero_de_cliente"                        
   [2] "foto_mes"                                 
   [3] "internet"                                 
   [4] "cliente_edad"                             
   [5] "cliente_antiguedad"                       
   [6] "mrentabilidad"                            
   [7] "mrentabilidad_annual"                     
   [8] "mcomisiones"                              
   [9] "mactivos_margen"                          
  [10] "mpasivos_margen"                          
  [11] "cproductos"                               
  [12] "mcuenta_corriente"                        
  [13] "mcaja_ahorro"                             
  [14] "cdescubierto_preacordado"                 
  [15] "mcuentas_saldo"                           
  [16] "ctarjeta_visa"                            
  [17] "ctarjeta_visa_transacciones"              
  [18] "mtarjeta_visa_consumo"                    
  [19] "ctarjeta_master"                          
  [20] "ctarjeta_master_transacciones"            
  [21] "mtarjeta_master_consumo"                  
  [22] "cprestamos_personales"                    
  [23] "mprestamos_personales"                    
  [24] "cpayroll_trx"                             
  [25] "mpayroll"                                 
  [26] "mttarjeta_visa_debitos_automaticos"       
  [27] "ccomisiones_mantenimiento"                
  [28] "mcomisiones_mantenimiento"                
  [29] "ccomisiones_otras"                        
  [30] "mtransferencias_recibidas"                
  [31] "ccallcenter_transacciones"                
  [32] "thomebanking"                             
  [33] "chomebanking_transacciones"               
  [34] "ctrx_quarter"                             
  [35] "Master_status"                            
  [36] "Master_mfinanciacion_limite"              
  [37] "Master_Fvencimiento"                      
  [38] "Master_msaldototal"                       
  [39] "Master_mlimitecompra"                     
  [40] "Master_fultimo_cierre"                    
  [41] "Master_fechaalta"                         
  [42] "Master_mconsumototal"                     
  [43] "Master_cconsumos"                         
  [44] "Master_mpagominimo"                       
  [45] "Visa_status"                              
  [46] "Visa_mfinanciacion_limite"                
  [47] "Visa_Fvencimiento"                        
  [48] "Visa_msaldototal"                         
  [49] "Visa_mlimitecompra"                       
  [50] "Visa_fultimo_cierre"                      
  [51] "Visa_fechaalta"                           
  [52] "Visa_mconsumototal"                       
  [53] "Visa_cconsumos"                           
  [54] "Visa_mpagominimo"                         
  [55] "clase_ternaria"                           
  [56] "kmes"                                     
  [57] "mpayroll_sobre_edad"                      
  [58] "entrenamiento"                            
  [59] "rf_001_006"                               
  [60] "rf_001_008"                               
  [61] "rf_001_007"                               
  [62] "rf_001_015"                               
  [63] "rf_001_001"                               
  [64] "rf_001_012"                               
  [65] "rf_001_000"                               
  [66] "rf_001_010"                               
  [67] "rf_001_011"                               
  [68] "rf_001_003"                               
  [69] "rf_001_009"                               
  [70] "rf_001_014"                               
  [71] "rf_001_005"                               
  [72] "rf_001_002"                               
  [73] "rf_001_013"                               
  [74] "rf_001_004"                               
  [75] "rf_002_009"                               
  [76] "rf_002_014"                               
  [77] "rf_002_002"                               
  [78] "rf_002_010"                               
  [79] "rf_002_006"      

#### 9.3.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr* por algun canal secreto de forma de no confundir a los *Analista Jr*  nni distraer con detalles operativos a la estratégica *Modalidad Gerencial*

In [45]:
# No se implementa la reduccion de la dimensionalidad con canaritos

### 9.3.2 Modelado

#### 9.3.2.1 Training Strategy

Se hace una estrategia de entrenamiento muy sencilla, tomando todos los meses posibles, SIN eliminar nada x pandemia ni por ningun otro motivo

* future = 202109  obviamente completo

* final_train =  [ 201901, 202107 ]  SIN undersampling

* training
   * testing = NO HAY
   * validation =  202107   completo, sin undersampling
   * training = [ 201901, 202105 ]  donde se consideran el 100% de los CONTINUA

In [46]:
PARAM$trainingstrategy$validate <- c(202105)

PARAM$trainingstrategy$training <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103
) # Corregido para el grupo B

PARAM$trainingstrategy$training_pct <- 1.0


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [47]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

In [48]:
# los campos en los que se entrena
campos_buenos <- copy( setdiff(
    colnames(dataset), c("clase_ternaria","clase01","azar"))
)

In [ ]:
# preparo para que se puede hacer undersampling de los CONTINUA
#  solamente por un tema de VELOCIDAD
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar:=runif(nrow(dataset))]

# undersampling de los CONTINUA
dataset[, fold_train :=  foto_mes %in%  PARAM$trainingstrategy$training &
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
     azar < PARAM$trainingstrategy$training_pct ) ]


if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

dtrain <- lgb.Dataset(
  data= data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
  label= dataset[fold_train == TRUE, clase01],
  free_raw_data= TRUE
)

In [49]:
# datos de validation
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
  label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
  free_raw_data= TRUE
)

nrow(dvalidate)

[1] 32900

####  9.3.2.2. Hyperparameter Tuning

* Clase binaria que se optimiza :  positivos = [ BAJA+1, BAJA+2 ]

* Metrica que se optimiza **AUC** Area Under Curve de la  ROC Curve

es muy importante notar que intencionalmente  **NO** se está optimizando la funcion de ganancia del problema

* Cantidad de iteraciones inteligentes de la Optimizacion Bayesiana = **10**

* Parametros no default, fijos de LightGBM que no se optimizan
  * max_bin = 31 , Alienigenas Ancestrales contruyeron las pirámides y dejaron a la humanidad en un jeroglifico  *max_bin=31*
  * feature_fraction = 0.5  para poner algo que generalmente no falla
  * learning_rate = 0.03  para que aprenda lento


* Parametros que se optimizan en la Bayesian Optimization
  * num_leaves  [8, 256]
  * min_data_in_leaf  [8, 8192]

In [50]:
# paquetes necesarios para la Bayesian Optimization
if(!require("DiceKriging")) install.packages("DiceKriging")
require("DiceKriging")

if(!require("mlrMBO")) install.packages("mlrMBO")
require("mlrMBO")

Loading required package: DiceKriging

Loading required package: mlrMBO

Loading required package: mlr

Loading required package: ParamHelpers


Attaching package: 'ParamHelpers'


The following object is masked from 'package:R.utils':

    isVector



Attaching package: 'mlr'


The following objects are masked from 'package:R.utils':

    resample, setThreshold


Loading required package: smoof

Loading required package: checkmate


Attaching package: 'checkmate'


The following object is masked from 'package:DiceKriging':

    checkNames


The following object is masked from 'package:R.utils':

    asInt



Attaching package: 'smoof'


The following objects are masked from 'package:R.oo':

    getDescription, getName




Definición de la Bayesian Optimization
<br> Si se desea optimizar un hiperparámetro que esta como fijo, debe QUITARSE de param_fijos y agregarse a PARAM$hipeparametertuning$hs

In [51]:
# un Analista Jr  debe poder animarse a hacer 100 iteraciones
PARAM$hipeparametertuning$num_interations <- 20

# parametros fijos del LightGBM
PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE, # para evitar warning
  seed= PARAM$semilla_primigenia,
  max_bin= 31,

  num_iterations= 9999,  # valor grande, lo limita early_stopping_rounds
  early_stopping_rounds= 400
)

PARAM$hipeparametertuning$hs <- makeParamSet(
  makeIntegerParam("num_leaves", lower= 2L, upper= 256L),
  makeIntegerParam("min_data_in_leaf", lower= 2L, upper= 8192L),
  makeNumericParam("feature_fraction", lower= 0.1, upper= 0.9),
  makeNumericParam("learning_rate", lower= 0.01, upper= 0.5)
)

Función "señora caja negra"  que es llamada para verificar la realidad por la Bayesian Optimization

In [52]:
# En  x llegan los parmaetros de la bayesiana
#  devuelve la AUC en validate del modelo entrenado
#  en el parametro x llegan los hiperparámetros que se estan optimizando

EstimarGanancia_AUC_lightgbm <- function(x) {

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # entreno LightGBM
  modelo_train <- lgb.train(
    data= dtrain,
    valids= list(valid = dvalidate),
    eval= "auc",
    param= param_completo,
    verbose= -100
  )

  # recupero la AUC en validation
  AUC <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]

  # esta es la forma de devolver un parametro extra
  attr(AUC, "extras") <- list("num_iterations"= modelo_train$best_iter)

  # hago espacio en la memoria
  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  message(format(Sys.time(), "%a %b %d %X %Y"), " AUC ", AUC)

  return(AUC)
}

seteo de la Bayesian Optimization (complejo)
<br> copiado y pegado de la documentación de la librería

In [54]:
configureMlr(show.learner.output = FALSE)

# configuro la busqueda bayesiana,  los hiperparametros que se van a optimizar
# por favor, no desesperarse por lo complejo
obj.fun <- makeSingleObjectiveFunction(
    fn= EstimarGanancia_AUC_lightgbm, # la funcion que voy a maximizar
    minimize= FALSE, # estoy Maximizando AUC
    noisy= FALSE,
    par.set= PARAM$hipeparametertuning$hs,
    has.simple.signature= FALSE # paso los parametros en una lista
)

# cada 600 segundos guardo el resultado intermedio
ctrl <- makeMBOControl(
    save.on.disk.at.time= 600,
    save.file.path= "HT.RDATA"
)

# indico la cantidad de iteraciones que va a tener la Bayesian Optimization
ctrl <- setMBOControlTermination(
    ctrl,
    iters= PARAM$hipeparametertuning$num_interations  # cantidad de iteraciones inteligentes
)

# defino el método estandar para la creacion de los puntos iniciales
#   los "No Inteligentes"
ctrl <- setMBOControlInfill(ctrl, crit = makeMBOInfillCritEI())

# mas configuraciones
surr.km <- makeLearner(
    "regr.km",
    predict.type= "se",
    covtype= "matern3_2",
    control= list(trace = TRUE)
)

Corrida de la Bayesian Optimization,  aqui se hace el trabajo pesado
<br> por favor no se asuste con los warnings que pudieran aparecer

Si corrío a medias y llegó a las iteraciones inteligentes, en el archivo binario HT.RDATA quedó lo ya procesado y es utilizado para retomar la corrida desde lo último que llegó a grabar.

In [55]:
# inicio la optimizacion bayesiana

if (!file.exists("HT.RDATA")) {
  bayesiana_salida <- mbo(obj.fun, learner= surr.km, control= ctrl)
} else {
  bayesiana_salida <- mboContinue("HT.RDATA") # retomo en caso que ya exista
}


Computing y column(s) for design. Not provided.

Tue Aug 12 2:15:24 PM 2025 AUC 0.918536855674742

Tue Aug 12 2:15:59 PM 2025 AUC 0.907675564380483

Tue Aug 12 2:17:08 PM 2025 AUC 0.921054990866608

Tue Aug 12 2:17:47 PM 2025 AUC 0.924018631509134

Tue Aug 12 2:18:46 PM 2025 AUC 0.920660509479948

Tue Aug 12 2:20:23 PM 2025 AUC 0.926191611406162

Tue Aug 12 2:21:07 PM 2025 AUC 0.909281761163912

Tue Aug 12 2:22:03 PM 2025 AUC 0.917714357950874

Tue Aug 12 2:23:26 PM 2025 AUC 0.92534007835802

Tue Aug 12 2:23:51 PM 2025 AUC 0.927223044606748

Tue Aug 12 2:25:58 PM 2025 AUC 0.904316890510339

Tue Aug 12 2:26:41 PM 2025 AUC 0.920432599164122

Tue Aug 12 2:28:16 PM 2025 AUC 0.926462353070222

Tue Aug 12 2:28:35 PM 2025 AUC 0.912695364434071

Tue Aug 12 2:29:32 PM 2025 AUC 0.886344321930092

Tue Aug 12 2:30:26 PM 2025 AUC 0.923639898814442

[mbo] 0: num_leaves=113; min_data_in_leaf=4971; feature_fraction=0.375; learning_rate=0.285 : y = 0.919 : 67.9 secs : initdesign

[mbo] 0: num_leaves=64

la bayesian optimization ha corrido, extraigo los mejores hiperparametros

In [56]:
# almaceno los resultados de la Bayesian Optimization
# y capturo los mejores hiperparametros encontrados

tb_bayesiana <- as.data.table(bayesiana_salida$opt.path)

# ordeno en forma descendente por AUC = y
setorder(tb_bayesiana, -y, -num_iterations)

# grabo para eventualmente poder utilizarlos en OTRA corrida
fwrite( tb_bayesiana,
  file="BO_log.txt",
  sep="\t"
)

# los mejores hiperparámetros son los que quedaron en el registro 1 de la tabla
PARAM$out$lgbm$mejores_hiperparametros <- tb_bayesiana[
  1, # el primero es el de mejor AUC
  list(num_leaves, min_data_in_leaf, num_iterations)
]

print(PARAM$out$lgbm$mejores_hiperparametros)

   num_leaves min_data_in_leaf num_iterations
        <int>            <int>          <int>
1:         60             3508            225


### 9.3.3 Produccion

#### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

##### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la optimización bayesiana

In [57]:
PARAM$trainingstrategy$final_train <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105
)  # corregido para el grupo B

dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

# creo el dfinal_train en formato  LightGBM
dfinal_train <- lgb.Dataset(
  data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
  label= dataset[fold_final_train == TRUE, clase01],
  free_raw_data= TRUE
)

nrow( dfinal_train) # verifico el tamaño

[1] 844168

##### Final Training Hyperparameters

In [58]:
# uno los parametros fijos y los mejores encontrados de los variables
fijos <- copy(PARAM$lgbm$param_fijos)

# quito lo que optimice en la Bayesian Optimization
fijos$num_iterations <- NULL
fijos$early_stopping_rounds <- NULL

# agrego a los hiperparametros fijos los que encontre con la Bayesian Optimization
param_final <- c(fijos, PARAM$out$lgbm$mejores_hiperparametros)

##### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

In [59]:
PARAM$FT$semillerio <- 20  # cantidad de semillas

In [60]:
if(!require("primes")) install.packages("primes")
require("primes")

Loading required package: primes



In [61]:
primos <- generate_primes(min = 100000, max = 1000000)
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
# me quedo con PARAM$semillerio  primos al azar
PARAM$FT$semillas <- sample(primos)[seq(PARAM$FT$semillerio)]

cat( PARAM$FT$semillas)

823591 119563 780799 800999 976211 378277 519067 557059 617387 761119 384533 172021 138113 969341 441157 333779 762659 172987 697979 118277

In [63]:
dir.create("modelos", showWarnings =FALSE)

primero <- TRUE
for( sem in PARAM$FT$semillas)
{
  nombre_arch <- paste0( "./modelos/modelo_", sem, ".txt")
  if( !file.exists(nombre_arch) )
  {
    param_final$seed <- sem

    set.seed(sem, kind = "L'Ecuyer-CMRG")
    final_model <- lgb.train(
      data= dfinal_train,
      param= param_final,
      verbose= -100
    )

    lgb.save(final_model, nombre_arch) # grabo el modelo"

    # grabo la primer importancia de variables
    #  Natalia : da lo mismo cual se guarda
    if( primero)
    {
      primero <- FALSE
      tb_importancia <- as.data.table(lgb.importance(final_model))
      archivo_importancia <- "impo.txt"

      fwrite( tb_importancia,
        file= archivo_importancia,
        sep= "\t"
      )
    }
 }
}


#### Scoring

Aplico el modelo final a los datos del futuro

In [64]:
PARAM$trainingstrategy$future <- c(202107) # ACÁ CAMBIAR LA FECHA PARA DATOS DEL FUTURO

dfuture <- dataset[ foto_mes %in% PARAM$trainingstrategy$future ]

In [65]:
# aplico final_model   a dfuture

tb_prediccion <- dfuture[, list(numero_de_cliente, clase_ternaria)] # modificado.
tb_prediccion[, prob := 0]

datos_matrix <- data.matrix(dfuture[, campos_buenos, with= FALSE])


for( isem in seq(length(PARAM$FT$semillas)) ) # Recorro cada semilla generada
{
   sem <- PARAM$FT$semillas[ isem ]
   nombre_arch <- paste0( "./modelos/modelo_", sem, ".txt") 
   final_model <- lgb.load(nombre_arch) # capturo el modelo construido con esa semilla
   # hago la prediccion con ese modelo 
   prediccion <- predict(
     final_model,
     datos_matrix
  )

  tb_prediccion[, paste0("prob_", isem) := prediccion]
  tb_prediccion[, prob := prob + prediccion]

  rm(final_model)
  rm(prediccion)
  gc(full = TRUE, verbose=FALSE)
}

rm( datos_matrix)
gc(full = TRUE, verbose=FALSE)

tb_prediccion[, prob := prob /length(PARAM$FT$semillas) ]

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2657274,142.0,4163028,222.4,4163028,222.4
Vcells,939641117,7168.9,2940126358,22431.4,2940119369,22431.4


##### Tabla Prediccion

In [66]:
# veo que hay en tb_prediccion
tb_prediccion

# grabo las probabilidad del modelo
#  me va a ser util para hacer Ensembles de modelos
fwrite(tb_prediccion,
  file= "prediccion.txt",
  sep= "\t"
)

numero_de_cliente,clase_ternaria,prob,prob_1,prob_2,prob_3,prob_4,prob_5,prob_6,prob_7,⋯,prob_11,prob_12,prob_13,prob_14,prob_15,prob_16,prob_17,prob_18,prob_19,prob_20
<int>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
29186441,CONTINUA,2.235774e-03,2.235774e-03,2.235774e-03,2.235774e-03,2.235774e-03,2.235774e-03,2.235774e-03,2.235774e-03,⋯,2.235774e-03,2.235774e-03,2.235774e-03,2.235774e-03,2.235774e-03,2.235774e-03,2.235774e-03,2.235774e-03,2.235774e-03,2.235774e-03
29187730,CONTINUA,1.400433e-04,1.400433e-04,1.400433e-04,1.400433e-04,1.400433e-04,1.400433e-04,1.400433e-04,1.400433e-04,⋯,1.400433e-04,1.400433e-04,1.400433e-04,1.400433e-04,1.400433e-04,1.400433e-04,1.400433e-04,1.400433e-04,1.400433e-04,1.400433e-04
29187961,CONTINUA,1.418172e-03,1.418172e-03,1.418172e-03,1.418172e-03,1.418172e-03,1.418172e-03,1.418172e-03,1.418172e-03,⋯,1.418172e-03,1.418172e-03,1.418172e-03,1.418172e-03,1.418172e-03,1.418172e-03,1.418172e-03,1.418172e-03,1.418172e-03,1.418172e-03
29193101,CONTINUA,1.548675e-03,1.548675e-03,1.548675e-03,1.548675e-03,1.548675e-03,1.548675e-03,1.548675e-03,1.548675e-03,⋯,1.548675e-03,1.548675e-03,1.548675e-03,1.548675e-03,1.548675e-03,1.548675e-03,1.548675e-03,1.548675e-03,1.548675e-03,1.548675e-03
29193281,CONTINUA,1.744897e-03,1.744897e-03,1.744897e-03,1.744897e-03,1.744897e-03,1.744897e-03,1.744897e-03,1.744897e-03,⋯,1.744897e-03,1.744897e-03,1.744897e-03,1.744897e-03,1.744897e-03,1.744897e-03,1.744897e-03,1.744897e-03,1.744897e-03,1.744897e-03
29198891,CONTINUA,5.059473e-04,5.059473e-04,5.059473e-04,5.059473e-04,5.059473e-04,5.059473e-04,5.059473e-04,5.059473e-04,⋯,5.059473e-04,5.059473e-04,5.059473e-04,5.059473e-04,5.059473e-04,5.059473e-04,5.059473e-04,5.059473e-04,5.059473e-04,5.059473e-04
29200651,CONTINUA,1.678504e-03,1.678504e-03,1.678504e-03,1.678504e-03,1.678504e-03,1.678504e-03,1.678504e-03,1.678504e-03,⋯,1.678504e-03,1.678504e-03,1.678504e-03,1.678504e-03,1.678504e-03,1.678504e-03,1.678504e-03,1.678504e-03,1.678504e-03,1.678504e-03
29200770,CONTINUA,5.976745e-05,5.976745e-05,5.976745e-05,5.976745e-05,5.976745e-05,5.976745e-05,5.976745e-05,5.976745e-05,⋯,5.976745e-05,5.976745e-05,5.976745e-05,5.976745e-05,5.976745e-05,5.976745e-05,5.976745e-05,5.976745e-05,5.976745e-05,5.976745e-05
29201701,CONTINUA,1.589493e-04,1.589493e-04,1.589493e-04,1.589493e-04,1.589493e-04,1.589493e-04,1.589493e-04,1.589493e-04,⋯,1.589493e-04,1.589493e-04,1.589493e-04,1.589493e-04,1.589493e-04,1.589493e-04,1.589493e-04,1.589493e-04,1.589493e-04,1.589493e-04


In [67]:
setorder(tb_prediccion, -prob)
top_2400 <- tb_prediccion[1:2400]

In [ ]:
library(data.table)
library(ggplot2)
library(scales)  # para formatear ejes

# Ordenar por probabilidad descendente
setorder(tb_prediccion, -prob)

# Rango de k hasta 2400
valores_k <- seq(1800, 2400, by = 100)


# Calcular ganancia para cada k
resultados <- data.table(k = valores_k)
resultados[, ganancia := sapply(k, function(topk) {
  top_k <- tb_prediccion[1:topk]
  top_k[, ganancia := ifelse(clase_ternaria == "BAJA+2", 117000, -3000)]
  sum(top_k$ganancia)
})]

# Graficar curva de ganancia con eje Y en miles
ggplot(resultados, aes(x = k, y = ganancia)) +
  geom_line(color = "steelblue", size = 1) +
  geom_point(color = "darkred") +
  geom_vline(xintercept = 2200, linetype = "dashed", color = "gray50") +
  scale_y_continuous(labels = label_number(scale = 1/1000, suffix = " K")) +
  labs(
    title = "Curva de Ganancia Acumulada (Analista Junior)",
    x = "Cantidad de envíos (k)",
    y = "Ganancia acumulada (miles)"
  ) +
  theme_minimal() +
  theme(
    plot.title = element_text(hjust = 0.5, size = 16, face = "bold"),
    axis.title = element_text(size = 14),
    axis.text = element_text(size = 12)
  )



  BAJA+1   BAJA+2 CONTINUA 
     179      161     2060 

In [ ]:
library(data.table)

# Ordenar por probabilidad descendente
setorder(tb_prediccion, -prob)

# Definir k (máximo)
k <- min(#NRO#, nrow(tb_prediccion))

# Seleccionar las primeras k filas
top_k <- tb_prediccion[1:k]

# Calcular ganancia
ganancia <- sum(ifelse(top_k$clase_ternaria == "BAJA+2", 117000, -3000))

# Formatear con separador de miles y coma decimal
ganancia_formateada <- format(
  round(ganancia, 2),
  big.mark = ".",
  decimal.mark = ",",
  nsmall = 2
)

# Conteos por clase
conteos <- top_k[, .N, by = clase_ternaria]

# Mostrar
cat("Ganancia total:", ganancia_formateada, "\n")
print(conteos)


[1] 12120000

clase_ternaria,N
<chr>,<int>
BAJA+1,179
CONTINUA,2060
BAJA+2,161


In [74]:
# grabo los parametros
if( !require("yaml")) install.packages("yaml")
require("yaml")

write_yaml( PARAM, file="PARAM.yml")

Loading required package: yaml



In [75]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Tue Aug 12 2:01:49 AM 2025"